# Model Finetuning — LogisticRegression GridSearchCV

Baseline model chosen in `03_feature_engineering_and_modeling.ipynb` (see `NOTES.md` Final Model Decision). This notebook tunes `LogisticRegression`'s hyperparameters via `GridSearchCV`. Prior EDA/feature engineering established a practical performance ceiling (~0.75 ROC-AUC / ~0.40 PR-AUC) — expect tuning to yield modest gains at best, but worth ruling out formally.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, recall_score,
    classification_report, confusion_matrix,
)

## Load data and rebuild final feature set

Reproduces the decisions from `NOTES.md`: drop `TotalCharges`/ratio duplicates and the 3 insignificant categorical columns, add `is_new_customer`. The interaction/ratio/composite features tried in `03_...` did not improve performance, so they're intentionally left out here.

In [ ]:
df = pd.read_csv("../data/processed/train.csv")

df.drop(columns=[
    "TotalCharges",
    "avg_charge_per_month(AccountAge)",
    "avg_charge_per_month(MonthlyCharges)",
    "DeviceRegistered",
    "MultiDeviceAccess",
    "PaperlessBilling",
    "MonthlyChargesGroup", "ViewingHoursPerWeekGroup", "AverageViewingDurationGroup",
    "ContentDownloadsPerMonthGroup", "UserRatingGroup", "SupportTicketsPerMonthGroup", "WatchlistSizeGroup",
], inplace=True, errors="ignore")


class NewCustomerFlagger(BaseEstimator, TransformerMixin):
    def __init__(self, account_age_col="AccountAge", threshold_months=12, output_col="is_new_customer"):
        self.account_age_col = account_age_col
        self.threshold_months = threshold_months
        self.output_col = output_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.output_col] = np.where(X[self.account_age_col] < self.threshold_months, "Yes", "No")
        return X

    def get_feature_names_out(self, input_features=None):
        input_features = list(input_features) if input_features is not None else []
        return np.array(input_features + [self.output_col])


df = NewCustomerFlagger(threshold_months=12).fit_transform(df)
df.shape

In [ ]:
categorical_cols = [
    "SubscriptionType", "PaymentMethod", "ContentType", "GenrePreference",
    "Gender", "ParentalControl", "SubtitlesEnabled", "is_new_customer",
]

numeric_cols_linear = [
    "AccountAge", "MonthlyCharges", "ViewingHoursPerWeek",
    "AverageViewingDuration", "ContentDownloadsPerMonth", "SupportTicketsPerMonth",
]

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numeric_cols_linear),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop",
)

## Train/test split

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

## GridSearchCV

Reduced grid for now (`l2`/`lbfgs`, 4 `C` values) — the full sweep across l1/l2/elasticnet penalties and saga/liblinear solvers crashed this machine. Re-run the full grid later on a more powerful machine. Scoring on PR-AUC (`average_precision`) — the metric used throughout `03_...` given class imbalance. `cv=3` to keep runtime reasonable given ~195K training rows.

In [ ]:
pipeline = Pipeline([
    ("preprocess", linear_preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])

# Lightweight grid for now — full sweep (l1/l2/elasticnet x saga/liblinear, more C values)
# crashed on this machine. Re-run the full grid on a more powerful machine later.
param_grid = {
    "clf__penalty": ["l2"],
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__solver": ["lbfgs"],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV PR-AUC:", grid_search.best_score_)

## Evaluate best model on held-out test set

In [ ]:
best_model = grid_search.best_estimator_

y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Test PR-AUC:", average_precision_score(y_test, y_proba))
print("\n", classification_report(y_test, y_pred))

print("\n--- baseline for comparison (untuned, from 03_...) ---")
print("roc_auc=0.7480, pr_auc=0.4004 (5-fold CV on X_train)")

## Conclusion: tuning does not move the needle

Business value at the profit-maximizing threshold (0.70) barely changed between the default-parameter model and the grid-searched model: `net_benefit_with_model` $121,600 (default) vs. $121,550 (tuned) — a ~$50 swing on a $4.4M baseline (~0.04%), i.e. noise, not a real regression.

This matches the practical performance ceiling (~0.75 ROC-AUC / ~0.40 PR-AUC) established in `03_...` — the dataset's information content, not the model's hyperparameters, is the limiting factor. **Keeping the default-parameter `LogisticRegression(class_weight="balanced")` as the final model** rather than the grid-searched one; see `NOTES.md` (Hyperparameter Tuning section) for the full writeup.